# Model Training
Train multiple models: ARIMA, XGBoost, LSTM

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))

PROCESSED_DATA_DIR = '../data/processed'

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

print('Starting model training')

Starting model training


## Load Data

In [2]:
df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'inflation_model_ready.csv'))
df['Date'] = pd.to_datetime(df['Date'])

# Remove rows with NaN target
df = df.dropna(subset=['Inflation_Rate_target'])

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')

Dataset shape: (22, 93)
Date range: 2005-01-01 00:00:00 to 2026-01-01 00:00:00


## Prepare Features and Target

In [3]:
# Features and target
X = df.drop(['Date', 'Inflation_Rate', 'Inflation_Rate_target'], axis=1)
y = df['Inflation_Rate_target']

# Time-based split (80-20)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'\nDate split: {df.iloc[split_idx]["Date"]}')

Training set: (17, 90)
Test set: (5, 90)

Date split: 2022-01-01 00:00:00


## Train XGBoost Model

In [4]:
# Train XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb_model.fit(X_train, y_train, verbose=False)
xgb_pred = xgb_model.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_r2 = r2_score(y_test, xgb_pred)

print(f'XGBoost Results:')
print(f'  RMSE: {xgb_rmse:.4f}')
print(f'  MAE: {xgb_mae:.4f}')
print(f'  R2: {xgb_r2:.4f}')

XGBoost Results:
  RMSE: 2.3104
  MAE: 2.3000
  R2: -0.9330


## Train Random Forest Model

In [5]:
# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print(f'Random Forest Results:')
print(f'  RMSE: {rf_rmse:.4f}')
print(f'  MAE: {rf_mae:.4f}')
print(f'  R2: {rf_r2:.4f}')

Random Forest Results:
  RMSE: 2.8485
  MAE: 2.7056
  R2: -1.9381


## Ensemble Model

In [6]:
# Simple ensemble: average predictions
ensemble_pred = (xgb_pred + rf_pred) / 2

ensemble_rmse = np.sqrt(mean_squared_error(y_test, ensemble_pred))
ensemble_mae = mean_absolute_error(y_test, ensemble_pred)
ensemble_r2 = r2_score(y_test, ensemble_pred)

print(f'Ensemble Results:')
print(f'  RMSE: {ensemble_rmse:.4f}')
print(f'  MAE: {ensemble_mae:.4f}')
print(f'  R2: {ensemble_r2:.4f}')

# Summary
print(f'\n=== Best Model ===')
print(f'XGBoost - RMSE: {xgb_rmse:.4f}')

Ensemble Results:
  RMSE: 2.5588
  MAE: 2.5028
  R2: -1.3709

=== Best Model ===
XGBoost - RMSE: 2.3104


## Save Models

In [7]:
# Save all models
model_dict = {
    'xgb': xgb_model,
    'rf': rf_model,
    'feature_names': list(X.columns),
    'metrics': {
        'xgb': {'rmse': xgb_rmse, 'mae': xgb_mae, 'r2': xgb_r2},
        'rf': {'rmse': rf_rmse, 'mae': rf_mae, 'r2': rf_r2},
        'ensemble': {'rmse': ensemble_rmse, 'mae': ensemble_mae, 'r2': ensemble_r2}
    }
}

model_path = os.path.join(PROCESSED_DATA_DIR, 'best_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model_dict, f)

print(f'✓ Models saved to: {model_path}')

✓ Models saved to: ../data/processed\best_model.pkl


In [8]:
# Test predictions on latest data
latest_features = X.iloc[-1:].values
xgb_latest_pred = xgb_model.predict(latest_features)[0]
rf_latest_pred = rf_model.predict(latest_features)[0]
ensemble_latest_pred = (xgb_latest_pred + rf_latest_pred) / 2

print(f'Predicted Inflation Rate (next year):')
print(f'  XGBoost: {xgb_latest_pred:.2f}%')
print(f'  Random Forest: {rf_latest_pred:.2f}%')
print(f'  Ensemble: {ensemble_latest_pred:.2f}%')

print(f'\nTraining Complete! ✓')

Predicted Inflation Rate (next year):
  XGBoost: 4.60%
  Random Forest: 5.29%
  Ensemble: 4.95%

Training Complete! ✓
